# wwd_i — Phases 3-4: train a wake-word head on Colab

Generates per-word data with **ElevenLabs v3**, embeds it through the **frozen `backbone.onnx`** already in the repo, trains a tiny **streaming GRU head**, calibrates the threshold to **FA < 0.5/hr & FR < 5%**, and exports `<word>_head.onnx`.

Change `WAKE_WORD` and re-run to train a different word — the backbone is reused; only the head is retrained.

A **GPU is optional** (the head is tiny; the backbone runs via ONNX Runtime). Set your **`ELEVENLABS_API_KEY`** in step 5.

### 1. (optional) GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'no GPU — fine, the head is tiny'

### 2. Get the code

`git clone` the repo (push it to GitHub first). For a private repo use `https://<TOKEN>@github.com/<you>/wwd_i.git`.

In [ ]:
import os

REPO_URL = "https://github.com/<you>/wwd_i.git"  # <-- set me
BRANCH = "master"
os.environ["REPO_URL"] = REPO_URL
os.environ["BRANCH"] = BRANCH
!rm -rf /content/wwd_i && git clone --branch "$BRANCH" --depth 1 "$REPO_URL" /content/wwd_i
%cd /content/wwd_i
!git log --oneline -1

### 3. Python 3.14 via uv

In [ ]:
!pip install -q uv
!uv venv --python 3.14
!.venv/bin/python --version

### 4. Install torch + the package + the ElevenLabs SDK

Head training is light; the frozen backbone runs through ONNX Runtime. `--torch-backend=auto` picks a CUDA wheel if a GPU is present, else CPU.

In [ ]:
!uv pip install --no-config --python .venv/bin/python torch torchaudio --torch-backend=auto
!uv pip install --python .venv/bin/python -e . onnxscript elevenlabs

### 5. Configure — wake word + API key

The key is read from the environment by the generator; it is never written to disk or code.

In [ ]:
import os

WAKE_WORD = "hey computer"  # <-- the phrase to detect
os.environ["ELEVENLABS_API_KEY"] = ""  # <-- paste your key
os.environ["WAKE_WORD"] = WAKE_WORD
os.environ["WORD_SLUG"] = WAKE_WORD.replace(" ", "_")
assert os.environ["ELEVENLABS_API_KEY"], "set ELEVENLABS_API_KEY above"
print("wake word:", WAKE_WORD)

## Part A — positives & hard negatives (ElevenLabs v3)

### A1. Smoke-test the SDK (one clip)

Validates your key and the SDK surface **before** spending on a batch. If it errors on the API call, the fix is usually a small tweak to `_synthesize`/`_client` in `wwd_i/data/elevenlabs.py` to match your installed SDK version.

In [ ]:
!.venv/bin/python -m wwd_i.data.elevenlabs --phrase "$WAKE_WORD" --out data/smoke --smoke

### A2. Generate positives (~300 diverse clips)

Many voices × prosody settings; cached, so re-running is cheap and bumping `--n-clips` only adds new variants.

In [ ]:
!.venv/bin/python -m wwd_i.data.elevenlabs --phrase "$WAKE_WORD" --out data/pos --n-clips 300

### A3. Generate hard negatives (near phrases)

The wake phrase's sub-words + generic confusables (`negatives.hard_negative_phrases`) — the main source of false triggers.

In [ ]:
!.venv/bin/python -m wwd_i.data.elevenlabs --hard-negs-for "$WAKE_WORD" --out data/hardneg --n-clips 20

## Part B — background noise & music

AudioSet (noise/general) + FMA (music) for additive augmentation and background negatives. Decoding is capped by `--max-bg`, so you don't need all of the files — just a diverse pool.

### B1. AudioSet — one balanced-train shard

In [ ]:
!mkdir -p data/bg/audioset
!wget -q -O /tmp/audioset.tar 'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/5a2fa42a1506470d275a47ff8e1fdac5b364e6ef/data/bal_train09.tar?download=true'
!tar -xf /tmp/audioset.tar -C data/bg/audioset && rm /tmp/audioset.tar
!echo "audioset files:"; find data/bg/audioset -type f | wc -l

### B2. (optional) FMA music — `fma_small` is ~7.5 GB; skip if bandwidth is tight

In [ ]:
# Optional music corpus. Comment the lines out to skip.
!mkdir -p data/bg/fma
!wget -q -O /tmp/fma_small.zip https://os.unil.cloud.switch.ch/fma/fma_small.zip
!unzip -q /tmp/fma_small.zip -d data/bg/fma && rm /tmp/fma_small.zip
!echo "fma mp3s:"; find data/bg/fma -name '*.mp3' | wc -l

## Part C — train the head

Embeds positives/negatives through the frozen backbone, trains the GRU, and calibrates the threshold. Watch the `[gate]` line: **PASS** = FA < 0.5/hr and FR < 5%.

`--n-bg-neg` sets the negative volume — more negatives sharpen the FA/hr estimate (and lengthen the embed step). You can also add your prepared **MSWC** dir to `--background` for speech negatives.

In [ ]:
!.venv/bin/python -m wwd_i.train.train_head --word "$WAKE_WORD" --positives data/pos --hard-neg data/hardneg --background data/bg --n-aug 5 --n-bg-neg 6000 --max-bg 3000 --epochs 40 --hidden 48 --out artifacts/${WORD_SLUG}_head.onnx --threshold-out artifacts/${WORD_SLUG}_head.json

## Part D — download the head + calibration

In [ ]:
from google.colab import files

slug = os.environ["WORD_SLUG"]
files.download(f"artifacts/{slug}_head.onnx")
files.download(f"artifacts/{slug}_head.json")

## Interpreting the gate

- **PASS** (FA < 0.5/hr, FR < 5%): drop `<word>_head.onnx` + its `.json` (threshold + refractory) into the repo — the frozen backbone plus this head are a complete detector → Phase 5 (streaming runtime).
- **High FA**: more / harder negatives (`--n-bg-neg`, add MSWC speech, more hard-neg phrases).
- **High FR**: more positives (`--n-clips`) or augmentation variety (`--n-aug`); confirm the A1/A2 clips are on-phrase.
- FA/hr is only as fine as the negative hours — scale `--n-bg-neg` to actually resolve < 0.5/hr.